# 2. Process SA BESS Data

This notebook regenerates the 100-household, one-year BESS aggregate from the selected signal definition.

## Setup

Load workflow helpers and either reuse diagnostic outputs or rerun diagnostics if needed.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

def find_publication_project(start: Path) -> Path:
    """Find publication/journal_article_1 from this moved notebook folder."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "scripts" / "process_pynnlf_output.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find publication/journal_article_1 from the current working directory.")

PROJECT_DIR = find_publication_project(Path.cwd())
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from sa_bess_publication_workflow import run_diagnostics_and_select, process_selected_sample, PROCESSED_DIR

## Select Profile And Households

The diagnostics function writes the report and returns the selected profile, selected households, and selected AEST window.

In [ ]:
selected_profile_id, selected_households, selection_info, metric_summary = run_diagnostics_and_select()
selected_households.head()

## Export Aggregate And Datasets

The 5-minute aggregate sums household power. The 30-minute PyNNLF datasets use arithmetic mean power.

In [ ]:
aggregate_5min, site_timeseries, household_summary = process_selected_sample(selected_profile_id, selected_households, selection_info)
print(aggregate_5min.shape)
print(PROCESSED_DIR)

## QA Snapshot

Check row counts, timestamp spacing, and selected household count.

In [ ]:
print('5-min rows:', len(aggregate_5min))
print('households:', household_summary['site_id'].nunique())
print('start:', aggregate_5min['datetime'].min())
print('end:', aggregate_5min['datetime'].max())
print('missing cells:', aggregate_5min.isna().sum().sum())